In [7]:
# !pip install pyspark==3.5.1

In [8]:
print(f"Pyspark-JupyterNotebook")

Pyspark-JupyterNotebook


In [9]:
import json
from dotenv import load_dotenv
import dotenv
from pyspark.sql import SparkSession
from IPython.display import HTML
import os
import sys

In [10]:
''' pip install python-dotenv'''
# load_dotenv() # will search for .env file in local folder and load variables 
# Reload the variables from your .env file, overriding existing ones
dotenv.load_dotenv(".env", override=True)

False

In [ ]:
# vi Jupyter_service.sh
# SPARK_PATH=/apps/monitoring_script/spark
# export PYSPARK_PYTHON=$SPARK_PATH/.venv/bin/python

'''
spark = SparkSession.builder.master("spark://localhost:7077").appName("PysparkJupyterNotebook").getOrCreate()

File "/apps/monitoring_script/spark/latest/python/lib/pyspark.zip/pyspark/worker.py", line 1104, in main
    "driver_version": str(version),
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 6) than that in driver 3.9, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set
'''
print(f"PYSPARK_PYTHON : {os.environ.get('PYSPARK_PYTHON')}, PYSPARK_DRIVER_PYTHON  : {os.environ.get('PYSPARK_DRIVER_PYTHON')}")

PYSPARK_PYTHON : c:\Users\euiyoung.hwang\Git_Workspace\python-fastapi-basic-k8s\.venv\Scripts\python.exe, PYSPARK_DRIVER_PYTHON  : None


In [12]:
print(sys.executable)

c:\Users\euiyoung.hwang\Git_Workspace\python-fastapi-basic-k8s\.venv\Scripts\python.exe


In [13]:
# os.environ["SPARK_HOME"] = "/c/Users/euiyoung.hwang/Git_Workspace/python-fastapi-basic-k8s/.venv/Lib/site-packages/pyspark"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = f'--jars /apps/monitoring_script/spark/jars/elasticsearch-spark-30_2.12-8.7.0.jar pyspark-shell'
''' restart Kernel on Jupyternotebook '''
''' A Jupyter kernel running a PySpark session might restart or crash for several reasons, primarily related to resource constraints (memory/CPU), code errors that block the kernel, or environment issues. In PySpark, managing memory is especially crucial due to its handling of large datasets.  '''

' A Jupyter kernel running a PySpark session might restart or crash for several reasons, primarily related to resource constraints (memory/CPU), code errors that block the kernel, or environment issues. In PySpark, managing memory is especially crucial due to its handling of large datasets.  '

In [14]:
def Spark_Json_test():
    ''' https://gibles-deepmind.tistory.com/entry/Docker-Container%EB%A1%9C-Spark%EA%B5%AC%EC%84%B1%ED%95%B4%EB%B3%B4%EA%B8%B0 '''
    # Initialize Spark Session (if not already done)
    '''
    # The number of CPU cores/slots per executor. 
    # For optimal performance in most environments, 5 cores per executor is widely recommended to balance HDFS throughput and CPU overhead
    # Example: 10 executors with 5 cores each = 50 total cores
    spark = SparkSession.builder \
        .appName("TotalCoreExample") \
        .config("spark.executor.instances", "10") \
        .config("spark.executor.cores", "5") \
        .getOrCreate()
    '''
    # spark = SparkSession.builder.master("spark://localhost:7077").appName("PysparkJupyterNotebook").getOrCreate()
    # spark = SparkSession.builder.appName("PysparkJupyterNotebook").config("spark.driver.memory", "1g").config("spark.executor.memory", "1g").getOrCreate()
    spark = SparkSession.builder.appName("PysparkJupyterNotebook").getOrCreate()
    
    # Create a sample DataFrame
    # df = spark.createDataFrame([(1, "Alice", 10), (2, "Bob", 20), (3, "Charlie", 30)], ["id", "name", "age"])
    # Create a DataFrame with sample data
    # data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
    # columns = ["Name", "Age"]
    # df = spark.createDataFrame(data, columns)
    data = [{"name": "John Doe", "age": 30, "id": "1"}, 
            {"name": "Jane Doe", "age": 25, "id": "2"}]
    df = spark.createDataFrame(data)

    # Increase partitions (shuffle happens)
    # repartitioned_df = df.repartition(20) # 20 partitions
    # Decrease partitions (less data shuffle)
    # df_coalesced = df.coalesce(5) # 5 partitions
    repartitioned_df = df.repartition(10)

    print(f"repartitioned_df.rdd.getNumPartitions() : {repartitioned_df.rdd.getNumPartitions()}")
    
    # Convert the DataFrame to an RDD of JSON strings, then collect into a Python list
    # json_strings = df.toJSON().collect()
    json_strings = repartitioned_df.toJSON().collect()
    
    # Pretty print the list of JSON objects using Python's json module
    for json_str in json_strings:
        # Use json.loads to convert the string to a Python object (dictionary)
        data = json.loads(json_str)
        # Use json.dumps with indent for pretty printing
        pretty_json = json.dumps(data, indent=4)
        print(pretty_json)


    def process_partition(iterator):
        # This function runs once per partition
        # 'iterator' contains all rows for that partition
        # Open a connection to an external system (e.g., database) once per partition
        # connection = open_db_connection() 
        print(f"Processing a new partition...")
        for row in iterator:
            # Perform operations on 'row'
            pass # Your logic here
        
        # Close the connection once the partition is processed
        # close_db_connection(connection)

    # In Apache Spark, the foreachPartition method is designed to execute your function in parallel across the worker nodes. 
    # When you call dataframe.foreachPartition(func), Spark automatically distributes the execution of func to run on each partition of the DataFrame simultaneously. 
    # Repartition the DataFrame for optimal parallelism and apply the function
    # df.repartition(20).foreachPartition(process_partition)
    # df.foreachPartition(process_partition)
    
    spark.stop()

In [15]:
Spark_Json_test()

repartitioned_df.rdd.getNumPartitions() : 10
{
    "age": 25,
    "id": "2",
    "name": "Jane Doe"
}
{
    "age": 30,
    "id": "1",
    "name": "John Doe"
}


In [16]:
HTML('<pre style="background-color: #A52A2A; color: white;">Job is being performed correctly</pre>')